In [ ]:
# @title 💤 Step 1: Anti-Sleep & Setup
import os
from IPython.display import display, Javascript
# यह कोड कोलाब को सोने से रोकेगा (Keep-Alive System)
display(Javascript('function ClickConnect(){document.querySelector("colab-connect-button").click()}setInterval(ClickConnect,60000)'))

print("⏳ लाइब्रेरीज़ इंस्टॉल हो रही हैं...")
!pip install -q gradio librosa soundfile coqui-tts torchcodec
from google.colab import drive
drive.mount('/content/drive')
os.makedirs("outputs", exist_ok=True)
print("✅ ड्राइव कनेक्ट हो गई और सेटअप तैयार है!")

In [ ]:
# @title 🚀 Step 2: ऐप लॉन्च करें (Drive + Unlimited Mode)
app_code = r'''
import gradio as gr
import torch, librosa, os, re, numpy as np, soundfile as sf
from TTS.api import TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_path = "/content/drive/MyDrive/VoiceBatchModels/"

print("⏳ सीधे ड्राइव से मॉडल लोड हो रहा है...")
tts = TTS(model_path=model_path, config_path=model_path + "config.json").to(device)

def voice_batch_engine(text, audio_sample, speed, pitch, sil_rem):
    if not audio_sample: return None
    final_output = 'outputs/VoiceBatch_Studio_Output.wav'
    parts = re.split(r'(?<=[।?!])\s+', text)
    combined_wav = []
    sr = 24000
    
    for p in parts:
        if len(p.strip()) < 2: continue
        temp_p = 'outputs/temp_p.wav'
        tts.tts_to_file(text=p, speaker_wav=audio_sample, language='hi', file_path=temp_p)
        y_p, _ = librosa.load(temp_p, sr=sr)
        combined_wav.extend(y_p)
    
    y = np.array(combined_wav)
    if sil_rem: y, _ = librosa.effects.trim(y, top_db=25)
    if speed != 1.0: y = librosa.effects.time_stretch(y, rate=speed)
    if pitch != 0: y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
    
    sf.write(final_output, y, sr)
    return final_output

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🎙️ VoiceBatch Studio v2.7.0 [Anti-Sleep Edition]')
    txt = gr.Textbox(label='Hindi Script (No Limit)', lines=10)
    smp = gr.Audio(label='Upload Sample', type='filepath')
    btn = gr.Button('Generate ⚡', variant='primary')
    out = gr.Audio(label='Download')
    btn.click(voice_batch_engine, [txt, smp, 1.0, 0, True], out)

demo.launch(share=True)
'''
with open('app.py', 'w') as f: f.write(app_code)
print("✅ ऐप लॉन्च हो रहा है...")
!python app.py